In [15]:
### Embeddings - Gemmini

# Etapas estudadas até agora:
# 1) Extração de textos em diferentes tipos de arquivos, como : txt,pdf, paginas web ....
# 2) Splittting em chunks (formato documento para subir pra vector Stores)
# 3) Conversão desses chunks em embeddings -> vetores  (Poderemos usar a Gemini, Ollama, e O hugginfFace)

In [12]:
pip install -q -U langchain-google-genai google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:


# chave de API usada, para manter secreta irei colocaar valor aleatorio depois de subir pro repo
os.environ["GOOGLE_API_KEY"] = "key_apagada_"

# Inicialize o modelo de embeddings do Gemini
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

# Testando a vetorização de um texto
vetor = embeddings.embed_query("Testando embeddings gratuitos com Gemini!")
print(f"Tamanho do vetor gerado: {len(vetor)}") 

In [21]:
print("""
    O modelo do Gemini embedding-2 pegou o meu texto e transformou em um vetor de 3072 features do tipo float
    Quando aplicado isso num banco vetorial, como por ex Pinecone, quando num sistema de RAG, o script usara esse vetor para calcular uma busca por similaridade
    Calculando a distancia matematica como a similaridade de cosseno com a pergunta que o user fizer
    Maior numero de dimensão, mas nuances e detalhes semanticos do texto, ambora ocupa mais memoria 
    Um menor numero de dimensões pode até ser mais rapido de calcular e menos espaço na memoria, porem, pode afetar no retorno da busca semantica

    Mas antes:
     > Ao fazermos o processo de vetorização dos chunks, é importante se preocupar como o slicing dos chunks é feito, evitando 'sujeiras' que atrabalham a semantica do chunk e posteriormente, a busca por similaridade

     Obs:
     > Na demonstração do vetor abaixo, foi puramente um processo de demonstração, e não uma aplicação na pratica
     > Projetos reais de RAG serão feitos nas aulas futuras 
     > Lembrando que, oque é vetorizado nos Documents (assim que textos extraidos são convertidos no formato ideal para salvar em vectorDB's, o que é vetorizado são os page.content) e nos metadados de cada chunk, vão os metadados de cada chunk dos documents
""")


    O modelo do Gemini embedding-2 pegou o meu texto e transformou em um vetor de 3072 features do tipo float
    Quando aplicado isso num banco vetorial, como por ex Pinecone, quando num sistema de RAG, o script usara esse vetor para calcular uma busca por similaridade
    Calculando a distancia matematica como a similaridade de cosseno com a pergunta que o user fizer
    Maior numero de dimensão, mas nuances e detalhes semanticos do texto, ambora ocupa mais memoria 
    Um menor numero de dimensões pode até ser mais rapido de calcular e menos espaço na memoria, porem, pode afetar no retorno da busca semantica

    Mas antes:
     > Ao fazermos o processo de vetorização dos chunks, é importante se preocupar como o slicing dos chunks é feito, evitando 'sujeiras' que atrabalham a semantica do chunk e posteriormente, a busca por similaridade

     Obs:
     > Na demonstração do vetor abaixo, foi puramente um processo de demonstração, e não uma aplicação na pratica
     > Projetos reais 

vetor

In [22]:
embeddings1024 = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    output_dimensionality=1024
)


embeddings_output = embeddings1024.embed_query(
    text="Testando texto para embeddings de 1024 dimnensoes"
)

print(len(embeddings_output))

print(embeddings_output)

1024
[-0.055264555, -0.011688092, 0.003979991, 0.006870602, 0.013858174, 0.03129707, 0.005232159, -0.0010081126, -0.012596171, -0.032563627, -0.012223213, -0.002110459, 0.025413416, 0.0030826172, 0.014110616, -0.0063979374, 0.03221358, -0.00822433, 0.006119349, 0.043892205, 0.006630533, 0.021117598, 0.014087326, 0.0010238093, -0.002654053, 0.020943707, 0.015583675, 0.008162678, -0.006025765, 0.19364986, -0.02737051, -0.01877398, -0.048085257, -0.0054832436, 0.030076815, -0.020038556, 0.020671478, -0.004743145, -0.008685741, 0.0068078567, 0.028879423, 0.037397824, 0.0031866704, 0.034572095, 0.02269818, 0.006140076, -0.029932195, 0.04218656, -0.008166185, -0.03722869, -0.03255803, 0.00021815197, -0.0023957412, -0.02254497, 0.011799739, -0.026972707, 0.012032871, -0.03147442, 0.030096777, 0.013771732, 0.01200495, -0.010210804, 0.018233525, -0.00601841, 0.015446078, 0.00036258664, 0.024079055, -0.030872526, 0.0077485912, -0.040599607, 0.013653014, -0.030175967, -0.0007140721, 0.01906968, -

In [23]:
###  ASSIM QUE PEGAMOS CHUNKERIZAMOS OS TEXTOS QUE EXTRAIMOS DOS DOCUMENTOS DESEJADOS
### CONVERTEMOS O PAGE.CONTENT DE CADA DOCUMENT E CONVERTEMOS ELES EM VETORES DO TIPO FLOAT PODENDO ASSUMIR VALORES +-
### DEVEMOS ARMAZENAR ESSES VETORES EM BANCOS VETORIAIS, NO CASO ABAIXO, VAMOS USAR O CHROMADB

In [71]:
pip install -q langchain-community langchain-chroma chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
import re
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
## 1 Extraindo texto de um documento do tipo txt, usarei um arquivo pequeno, pois, estou usando o free tier do modelo de embeddings do gemini

In [70]:
document = TextLoader(
    file_path=r"C:\Users\Mateus\Desktop\Udemy\Complete Agentic AI Bootcamp\Secao5 - IngestãoDados\Documentos\Ai_Agents.txt.txt",
    encoding="utf-8"
)


document_loader = document.load()



# 2. Limpeza profunda do texto
for doc in document_loader:
    texto = doc.page_content.replace("\n", " ")
    # Remove sujeiras como  === ou ---, oque atrapala no calculo de similaridade
    texto = re.sub(r'={2,}|-{2,}', '', texto)
    # Substitui  espaços consecutivos por um só
    doc.page_content = re.sub(r'\s+', ' ', texto).strip()


split_document = RecursiveCharacterTextSplitter(
    separators=[" 1.", " 2.", " 3.", " 4.", " 5.", " 6.", " "],
    chunk_size=500,
    chunk_overlap=0
)


chunks = split_document.split_documents(
    document_loader
)



# só pra debugar como estão organizados
for i in range(0,len(chunks)):
    print(f'Chunk numero -> {i}')
    print(chunks[i].page_content)
    print("///////////////")
    print('\n')

Chunk numero -> 0
A REVOLUÇÃO DOS AGENTES DE INTELIGÊNCIA ARTIFICIAL
///////////////


Chunk numero -> 1
1. O Conceito de Agente de IA - Diferente dos modelos de linguagem tradicionais (LLMs) que apenas geram texto, - um Agente de IA é um sistema autônomo projetado para perceber o ambiente, - tomar decisões fundamentadas e executar ações para atingir metas específicas. - O foco principal deixa de ser a simples resposta e passa a ser a execução.
///////////////


Chunk numero -> 2
2. A Arquitetura Fundamental - A espinha dorsal de um agente é composta por quatro pilares essenciais: * Percepção: capacidade de ler dados, textos, imagens e contextos. * Cérebro (LLM): o modelo que raciocina, planeja e decide o próximo passo. * Ferramentas (Tools): integração com APIs, bancos de dados e navegadores. * Memória: armazenamento de curto prazo (contexto) e longo prazo (vetores).
///////////////


Chunk numero -> 3
3. Como Funciona o Ciclo de Ação (ReAct) - O padrão ReAct (Reasoning and Acting) gu

In [73]:
# por mais que eu passei o chunks com os metadados de cada chunk, quando ele for passar pro nosso embeddings1024 que é o nosso modelo de embeddings com 1024 dimenões, 
# o chromaDB só vai salvar o page content que agora só são valores float

db = Chroma.from_documents(
    chunks,
    embeddings1024
)

In [74]:
db

In [87]:
## aqui eu montei a pergunta que quero descobrir usando o calculo de similaridade
query = """
Cite as principais aplicações práticas de usar agentes de IA no mundo real

"""

# k = 1 numero de chunks retornados, o primeiro que aparece é oque esta no topo do rank de similiridade
retrieved_results = db.similarity_search(query,k=1)

print(retrieved_results[0].page_content)

5. Aplicações Práticas no Mundo Real - Automação de suporte ao cliente com resolução ativa de problemas. - Análise financeira automatizada e geração de relatórios em tempo real. - Pesquisa de mercado autônoma com varredura da web e compilação de dados. - Desenvolvimento de software, auxiliando no refatoramento e depuração de código.
